In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import pyspark.sql.functions as F

In [ ]:
spark = SparkSession.builder.appName("Airbnb London£").getOrCreate()

In [ ]:
listings = spark.read.csv(
    "../data-src/listings.csv.gz",
    header=True,
    inferSchema=True,
    sep=",",
    quote='"',
    escape='"',
    multiLine=True,
    mode="PERMISSIVE",
)
listings.show(5)

In [ ]:
for prop in listings.schema:
    print(prop)

In [ ]:
neighbourhoods = listings.select(listings.neighbourhood_cleansed)
neighbourhoods.show(10, truncate=False)

In [ ]:
high_score_listings = listings \
    .filter((listings.review_scores_location > 4.5) & (listings.number_of_reviews > 100)) \
    .filter(listings.price.isNotNull()) \
    .select("id", "name", "price", "review_scores_location", "number_of_reviews")

high_score_listings.show(20, truncate=False)

In [ ]:
from pyspark.sql.functions import regexp_replace

good_deals_df = high_score_listings.withColumn(
    "price_num", regexp_replace("price", "[$,]", "").cast("float")
) \
.filter(col('price_num') < 100) \
.sort(
    ["price_num", "review_scores_location", "number_of_reviews"],
    ascending=[True, False, False],
)

good_deals_df.schema["price_num"]

In [ ]:
good_deals_df.show(20, truncate=False)

In [ ]:
good_deals_df.count()

In [ ]:
# good_deals_df.write.format("csv").save("good_deals.csv")

In [ ]:
listings.groupby('source').count().show(truncate=False)

In [ ]:
listings.groupby('source').agg(
    F.count("source").alias("Count by Source"),
    F.round(F.sum("reviews_per_month"), 2).alias("Total Monthly Reviews"),
    F.round(F.avg("reviews_per_month"), 2).alias("Avg Monthly Review")
).show(truncate=False)

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

In [ ]:
@udf(StringType())
def Initial(name):
    return name[0]

In [ ]:
draft = listings.select("source", "name")

In [ ]:
draft.withColumn("initial",Initial(draft['name'])).show(3)

In [ ]:
# spark.stop()